# 3.5 — Linear Regression (OLS)

Linear regression with ordinary least squares (OLS) chooses coefficients for a straight-line rule by making squared residuals as small as possible. In this lesson, you will build the design matrix, residuals, empirical risk, normal equation, validation comparison, and stability penalty from scratch in NumPy so the fitted line is never a black box.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build OLS one idea at a time. Run each cell in order and read the printed intermediate values — every formula is turned into arrays, products, losses, and plots you can inspect. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, matrix multiplication, and linear algebra for OLS.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for any random-looking examples.

### 1. The design matrix: turning a line into matrix columns

A simple line has the form $\hat y=\beta_0+\beta_1x$. OLS stores that rule in a **design matrix** $X$: the first column is all ones for the intercept $\beta_0$, and the second column is the feature $x$ for the slope $\beta_1$. Once the columns are built, every prediction is the same matrix product $X\beta$.

In [ ]:
x_w = np.array([0., 1., 2., 3., 4.])  # one feature: study hours.
y_w = np.array([1.1, 2.0, 2.9, 4.1, 5.0])  # target: quiz score.
X_w = np.column_stack([np.ones_like(x_w), x_w])  # intercept column + slope column.
print("X shape:", X_w.shape)  # 5 examples x 2 coefficients.
print("design matrix:\n", X_w)  # inspect the two columns used by the line.

▶ What you'll see: the first column is all 1s, so the intercept can shift the line up or down.

In [ ]:
beta_guess_w = np.array([0.8, 1.0])  # a guessed intercept and slope.
yhat_guess_w = X_w @ beta_guess_w  # predictions from the matrix rule.
print("guess beta:", beta_guess_w)  # inspect the current coefficients.
print("predictions:", np.round(yhat_guess_w, 2))  # one fitted value per row of X.

▶ What you'll see: multiplying `X_w @ beta_guess_w` evaluates the whole line on all five x-values at once.

In [ ]:
plt.figure(figsize=(4.4, 3.2))  # compact plot of data and guessed line.
plt.scatter(x_w, y_w, color="black", label="observed y")  # actual training data.
plt.plot(x_w, yhat_guess_w, color="teal", label="guess line")  # fitted values from the guess.
plt.xlabel("x"); plt.ylabel("y"); plt.title("1: a line as Xβ")
plt.legend(); plt.show()

▶ What you'll see: the guessed line is close, but the dots are not exactly on it — those vertical gaps become residuals.

*Why it's done this way:* the intercept is not magic; it is just a column of ones. Writing the line as $X\beta$ lets the same linear-algebra machinery handle one feature, many features, or engineered columns without changing the prediction rule.

### 2. Residuals and empirical risk: what OLS minimizes

A residual is the signed miss $e_i=y_i-\hat y_i$. OLS squares each residual, then averages or sums those squared misses. Squaring makes every miss nonnegative, punishes large misses more than small misses, and creates a smooth bowl-shaped objective that has a clean minimum.

In [ ]:
resid_guess_w = y_w - yhat_guess_w  # signed vertical gaps from dots to the guessed line.
sq_guess_w = resid_guess_w ** 2  # squared residuals used by OLS.
print("residuals:", np.round(resid_guess_w, 3))  # positive = line too low, negative = too high.
print("squared residuals:", np.round(sq_guess_w, 3))  # all nonnegative.

▶ What you'll see: signs disappear after squaring, while larger misses contribute disproportionately.

In [ ]:
risk_guess_w = float(np.mean(sq_guess_w))  # empirical risk as average squared loss.
print("mean squared error:", round(risk_guess_w, 3))  # the raw training score.
assert round(risk_guess_w, 3) == 0.054  # concrete check for this guessed line.

▶ What you'll see: the guessed line's average squared loss is small but not zero.

In [ ]:
slope_grid_w = np.linspace(0.6, 1.3, 80)  # try many slopes while keeping intercept fixed.
risk_grid_w = []  # store MSE for each slope.
for slope_w in slope_grid_w:
    pred_w = X_w @ np.array([0.8, slope_w])  # evaluate this slope choice.
    risk_grid_w.append(float(np.mean((y_w - pred_w) ** 2)))  # record MSE.
plt.figure(figsize=(4.4, 3.2))
plt.plot(slope_grid_w, risk_grid_w, color="purple")  # one-dimensional loss bowl.
plt.axvline(beta_guess_w[1], color="gray", linestyle="--", label="guess slope")
plt.xlabel("slope with intercept fixed at 0.8"); plt.ylabel("MSE")
plt.title("2: squared-error bowl"); plt.legend(); plt.show()

▶ What you'll see: the loss forms a bowl over slopes, so OLS can look for the bottom rather than chase signs.

*Why it's done this way:* ERM asks for the hypothesis with the smallest average training loss. OLS makes that loss squared residual error, so the fitting problem becomes a smooth geometric search for the coefficient vector whose predictions are closest to $y$.

### 3. The normal equation: solving for the best coefficients

For squared loss, setting the gradient of $\lVert y-X\beta\rVert^2$ to zero gives $X^\top X\hat\beta=X^\top y$. When $X^\top X$ is invertible, the solution is $\hat\beta=(X^\top X)^{-1}X^\top y$. This is the normal equation from the lesson prose.

In [ ]:
XtX_w = X_w.T @ X_w  # the coefficient-side curvature matrix.
Xty_w = X_w.T @ y_w  # the target projected onto each column of X.
print("X^T X:\n", XtX_w)  # inspect the 2x2 system matrix.
print("X^T y:", np.round(Xty_w, 3))  # inspect the right-hand side.

▶ What you'll see: OLS reduces five data rows to a 2×2 linear system for intercept and slope.

In [ ]:
beta_hat_w = np.linalg.solve(XtX_w, Xty_w)  # solve instead of explicitly forming an inverse.
yhat_w = X_w @ beta_hat_w  # fitted values from the OLS coefficients.
resid_w = y_w - yhat_w  # final residuals.
print("beta_hat:", np.round(beta_hat_w, 3))  # intercept and slope.
print("fitted values:", np.round(yhat_w, 3))  # predictions at the training x-values.
assert np.allclose(np.round(beta_hat_w, 3), [1.040, 0.990])  # verified OLS coefficients.

▶ What you'll see: the fitted line is roughly $\hat y=1.04+0.99x$.

In [ ]:
orth_w = X_w.T @ resid_w  # residuals should be orthogonal to every column of X.
print("X^T residuals:", np.round(orth_w, 10))  # near zero means no linear signal remains.
assert np.allclose(orth_w, np.zeros(2), atol=1e-10)  # normal-equation check.
plt.figure(figsize=(4.4, 3.2))
plt.scatter(x_w, y_w, color="black", label="observed")
plt.plot(x_w, yhat_w, color="crimson", label="OLS fit")
for xi_w, yi_w, pi_w in zip(x_w, y_w, yhat_w):
    plt.plot([xi_w, xi_w], [pi_w, yi_w], color="gray", linewidth=1)  # residual segment.
plt.xlabel("x"); plt.ylabel("y"); plt.title("3: OLS residuals after solving")
plt.legend(); plt.show()

▶ What you'll see: vertical residual segments remain, but their pattern has no leftover intercept or slope direction.

*Why it's done this way:* at the optimum, nudging the prediction along any column of $X$ cannot reduce squared error. Algebraically that condition is $X^\top(y-X\hat\beta)=0$, which is exactly the normal equation.

### 4. Training score, cost, and the full decision score

The lesson's toy arithmetic reminds us that the raw training average is not always the final model-selection number. We compute the empirical average from three verified losses, add a method cost of 0.100, and compare against a tempting alternative. The important habit is to compare scores on the **same scale**.

In [ ]:
losses_w = np.array([0.235, 0.083, 0.454])  # verified per-example losses from the lesson text.
R_S_w = float(np.mean(losses_w))  # empirical risk average.
cost_w = 0.100  # complexity, regularization, or operational cost.
score_w = round(R_S_w, 3) + cost_w  # full selection score using the lesson-rounded empirical risk.
print("R_S:", round(R_S_w, 3), "cost:", round(cost_w, 3), "score:", round(score_w, 3))
assert round(R_S_w, 3) == 0.257 and round(score_w, 3) == 0.357  # lesson numbers.

▶ What you'll see: the raw average 0.257 becomes 0.357 after including the cost term.

In [ ]:
alternative_w = 0.393  # score for a more flexible alternative.
gap_w = alternative_w - score_w  # absolute gap.
relative_gap_w = gap_w / alternative_w  # gap on the alternative's scale.
print("gap:", round(gap_w, 3), "relative gap:", round(relative_gap_w, 3))
assert round(gap_w, 3) == 0.036 and round(relative_gap_w, 3) == 0.092  # lesson numbers.

▶ What you'll see: the baseline is lower, but the absolute gap is only 0.036.

In [ ]:
plt.figure(figsize=(4.6, 3.2))
plt.bar(["raw R_S", "+ cost score", "alternative"], [R_S_w, score_w, alternative_w], color=["gray", "teal", "orange"])
plt.ylabel("decision score"); plt.title("4: raw fit is not the full score")
plt.show()

▶ What you'll see: the raw training number looks best only because it omits the cost included in the selection score.

*Why it's done this way:* selection should optimize the quantity implied by the method, not the prettiest fragment. If a cost or complexity term is part of the rule, dropping it silently changes the algorithm and can choose a brittle model.

### 5. Stabilization, validation, and the final model choice

A stabilizing knob such as regularization trades a little flexibility for a score that is less likely to vanish on new data. In the lesson arithmetic, a 20% reduction changes 0.357 into 0.286. The final decision compares baseline, flexible alternative, and stabilized score with one consistent rule: lower is better.

In [ ]:
stable_w = 0.80 * score_w  # a 20% reduction from the stabilizing knob.
candidates_w = np.array([score_w, alternative_w, stable_w])  # all comparable scores.
labels_w = np.array(["baseline", "flexible", "stabilized"])
best_idx_w = int(np.argmin(candidates_w))  # choose the lowest full score.
print("stable score:", round(stable_w, 3))  # 0.286.
print("best model:", labels_w[best_idx_w], "with score", round(float(candidates_w[best_idx_w]), 3))
assert round(stable_w, 3) == 0.286 and labels_w[best_idx_w] == "stabilized"  # lesson decision.

▶ What you'll see: the stabilized score is the minimum among the three choices.

In [ ]:
train_mse_w = float(np.mean(resid_w ** 2))  # OLS training MSE from the fitted line.
x_val_w = np.array([0.5, 1.5, 2.5, 3.5])  # future-like validation x-values.
y_val_w = np.array([1.55, 2.50, 3.45, 4.55])  # validation targets not used in fitting.
X_val_w = np.column_stack([np.ones_like(x_val_w), x_val_w])  # validation design matrix.
val_mse_w = float(np.mean((y_val_w - X_val_w @ beta_hat_w) ** 2))  # validation risk.
print("train MSE:", round(train_mse_w, 4), "validation MSE:", round(val_mse_w, 4))
assert round(train_mse_w, 4) == 0.0054 and round(val_mse_w, 4) == 0.0018  # verified numbers.

▶ What you'll see: the validation score is computed on examples that were not used in the normal-equation fit.

In [ ]:
plt.figure(figsize=(4.8, 3.2))
plt.bar(labels_w, candidates_w, color=["teal", "orange", "seagreen"])
plt.ylabel("full decision score"); plt.title("5: compare on one scale")
plt.show()

▶ What you'll see: the final comparison is visual and numerical: stabilized is lowest, flexible is highest.

*Why it's done this way:* OLS is an optimizer, not a guarantee of future performance. Validation checks whether the optimized training rule survives unseen data, and stabilization gives up brittle variation when the full decision score says that trade is worthwhile.

## 🛠️ Setup

In [ ]:
import numpy as np # Import NumPy for arrays, matrix multiplication, least-squares solves, and numerical checks.
import matplotlib.pyplot as plt # Import Matplotlib so each OLS idea can be inspected visually.
np.random.seed(0) # Fix the global random seed so every stochastic example is repeatable.

## 🟢 Basics (warm-up)

### Basic 1 — Build a design matrix

**Goal.** Turn scalar x-values into a matrix with an intercept column, because OLS coefficients multiply columns. We build it in 2 steps.

In [ ]:
x_b1 = np.array([0., 1., 2., 3.]) # Define four simple feature values.
y_b1 = np.array([1., 2., 3., 4.]) # Define matching targets for inspection.
X_b1 = np.column_stack([np.ones_like(x_b1), x_b1]) # Add the intercept column of ones beside x.
print("X_b1:\n", X_b1) # Inspect the matrix that will be multiplied by beta.

▶ What you'll see: a two-column matrix where column 0 fits the intercept and column 1 fits the slope.

In [ ]:
plt.figure(figsize=(4, 3)) # Create a compact plot.
plt.scatter(x_b1, y_b1, color="black") # Show the observed points before fitting.
plt.title("Basic 1: raw points before OLS") # Title the warm-up plot.
plt.xlabel("x") # Label the feature axis.
plt.ylabel("y") # Label the target axis.
plt.show() # Display the scatter plot.

▶ What you'll see: the points form an obvious line, making the design-matrix columns easy to reason about.

👀 Takeaway: OLS starts by expressing the prediction rule as a matrix product $X\beta$.

### Basic 2 — Predict from chosen coefficients

**Goal.** Multiply a design matrix by a coefficient vector, because fitted values are what residuals compare against. We build it in 2 steps.

In [ ]:
x_b2 = np.array([0., 1., 2., 3.]) # Define x-values.
X_b2 = np.column_stack([np.ones_like(x_b2), x_b2]) # Build the intercept-plus-slope design matrix.
beta_b2 = np.array([1.0, 0.5]) # Choose intercept 1.0 and slope 0.5.
print("beta:", beta_b2) # Inspect the rule yhat = 1 + 0.5x.

▶ What you'll see: the coefficient vector has one number per column of X.

In [ ]:
yhat_b2 = X_b2 @ beta_b2 # Compute predictions for every row.
print("predictions:", yhat_b2) # Inspect fitted values.
assert np.allclose(yhat_b2, [1.0, 1.5, 2.0, 2.5]) # Verify the matrix multiplication result.
plt.figure(figsize=(4, 3)) # Create a compact line plot.
plt.plot(x_b2, yhat_b2, marker="o", color="teal") # Draw predictions from the chosen beta.
plt.title("Basic 2: predictions from Xβ") # Title the plot.
plt.xlabel("x") # Label x.
plt.ylabel("predicted y") # Label predictions.
plt.show() # Display the fitted line.

▶ What you'll see: the prediction rises by 0.5 whenever x increases by 1.

👀 Takeaway: coefficients are not abstract; each one scales a concrete design-matrix column.

### Basic 3 — Compute residuals

**Goal.** Calculate $y-\hat y$, because residuals are the signed errors that OLS squares. We build it in 2 steps.

In [ ]:
x_b3 = np.array([0., 1., 2., 3.]) # Define feature values.
y_b3 = np.array([1.2, 1.7, 2.1, 2.9]) # Define observed targets.
X_b3 = np.column_stack([np.ones_like(x_b3), x_b3]) # Build X.
beta_b3 = np.array([1.0, 0.6]) # Choose a candidate line.
yhat_b3 = X_b3 @ beta_b3 # Compute fitted values.
print("observed:", y_b3) # Inspect y.
print("predicted:", yhat_b3) # Inspect yhat.

▶ What you'll see: predictions are close but not identical to observations.

In [ ]:
resid_b3 = y_b3 - yhat_b3 # Compute signed residuals.
print("residuals:", np.round(resid_b3, 3)) # Inspect vertical gaps.
assert np.allclose(np.round(resid_b3, 3), [0.2, 0.1, -0.1, 0.1]) # Verify signs and values.
plt.figure(figsize=(4, 3)) # Create a residual plot.
plt.axhline(0, color="black", linewidth=1) # Add the zero-error line.
plt.bar(range(len(resid_b3)), resid_b3, color="orange") # Show signed residuals.
plt.title("Basic 3: signed residuals") # Title the plot.
plt.xlabel("example") # Label examples.
plt.ylabel("y - yhat") # Label residuals.
plt.show() # Display residual bars.

▶ What you'll see: bars above zero mean the line under-predicted; bars below zero mean it over-predicted.

👀 Takeaway: residual signs are useful diagnostics, but OLS minimizes their squared sizes.

### Basic 4 — Square residuals and average them

**Goal.** Compute mean squared error, because it is the empirical risk minimized by OLS. We build it in 3 steps.

In [ ]:
resid_b4 = np.array([0.2, 0.1, -0.1, 0.1]) # Reuse a tiny residual vector.
sq_b4 = resid_b4 ** 2 # Square each residual.
print("squared residuals:", np.round(sq_b4, 3)) # Inspect nonnegative losses.

▶ What you'll see: the negative residual becomes a positive loss just like the positive ones.

In [ ]:
mse_b4 = float(np.mean(sq_b4)) # Average the per-example squared losses.
print("MSE:", round(mse_b4, 4)) # Inspect the empirical risk.
assert round(mse_b4, 4) == 0.0175 # Verify the average.

▶ What you'll see: the average squared miss summarizes the whole residual vector as one score.

In [ ]:
plt.figure(figsize=(4, 3)) # Create a compact loss plot.
plt.bar(["e0²", "e1²", "e2²", "e3²"], sq_b4, color="purple") # Show each squared error.
plt.axhline(mse_b4, color="red", linestyle="--", label="MSE") # Add average line.
plt.title("Basic 4: squared losses") # Title the plot.
plt.legend() # Show the MSE label.
plt.show() # Display the loss bars.

▶ What you'll see: the dashed line is the empirical risk OLS tries to reduce.

👀 Takeaway: OLS turns many signed errors into one smooth average loss.

### Basic 5 — Solve the normal equation

**Goal.** Fit intercept and slope with $X^TX\beta=X^Ty$, because that is the OLS optimum for squared loss. We build it in 3 steps.

In [ ]:
x_b5 = np.array([0., 1., 2., 3., 4.]) # Define feature values.
y_b5 = np.array([1.1, 2.0, 2.9, 4.1, 5.0]) # Define targets.
X_b5 = np.column_stack([np.ones_like(x_b5), x_b5]) # Build design matrix.
XtX_b5 = X_b5.T @ X_b5 # Compute X^T X.
Xty_b5 = X_b5.T @ y_b5 # Compute X^T y.
print("X^T X:\n", XtX_b5) # Inspect the normal-equation matrix.

▶ What you'll see: a 2×2 system summarizes the intercept and slope fit.

In [ ]:
beta_b5 = np.linalg.solve(XtX_b5, Xty_b5) # Solve for OLS coefficients.
print("beta_hat:", np.round(beta_b5, 3)) # Inspect intercept and slope.
assert np.allclose(np.round(beta_b5, 3), [1.040, 0.990]) # Verify fitted coefficients.

▶ What you'll see: the best line is approximately 1.04 + 0.99x.

In [ ]:
yhat_b5 = X_b5 @ beta_b5 # Compute fitted values.
plt.figure(figsize=(4, 3)) # Create fit plot.
plt.scatter(x_b5, y_b5, color="black", label="data") # Plot observations.
plt.plot(x_b5, yhat_b5, color="crimson", label="OLS") # Plot fitted line.
plt.title("Basic 5: normal-equation fit") # Title plot.
plt.xlabel("x") # Label x.
plt.ylabel("y") # Label y.
plt.legend() # Show labels.
plt.show() # Display fit.

▶ What you'll see: the line balances the dots rather than passing through every point.

👀 Takeaway: the normal equation gives the exact least-squares coefficients for this full-rank design.

### Basic 6 — Check residual orthogonality

**Goal.** Verify $X^T(y-X\hat\beta)=0$, because OLS leaves no residual signal in any design column. We build it in 2 steps.

In [ ]:
x_b6 = np.array([0., 1., 2., 3., 4.]) # Define feature values.
y_b6 = np.array([1.1, 2.0, 2.9, 4.1, 5.0]) # Define targets.
X_b6 = np.column_stack([np.ones_like(x_b6), x_b6]) # Build design matrix.
beta_b6 = np.linalg.solve(X_b6.T @ X_b6, X_b6.T @ y_b6) # Fit OLS.
resid_b6 = y_b6 - X_b6 @ beta_b6 # Compute residuals after fitting.
print("residuals:", np.round(resid_b6, 3)) # Inspect remaining errors.

▶ What you'll see: residuals still exist because the data are not perfectly collinear.

In [ ]:
orth_b6 = X_b6.T @ resid_b6 # Project residuals onto intercept and x columns.
print("X^T residual:", np.round(orth_b6, 10)) # Inspect near-zero projections.
assert np.allclose(orth_b6, np.zeros(2), atol=1e-10) # Verify OLS first-order condition.
plt.figure(figsize=(4, 3)) # Create diagnostic plot.
plt.bar(["ones column", "x column"], orth_b6, color="seagreen") # Show projections.
plt.title("Basic 6: no leftover column signal") # Title plot.
plt.ylabel("X column · residual") # Label projection.
plt.show() # Display bars.

▶ What you'll see: both bars are essentially zero, which is the geometric signature of the OLS optimum.

👀 Takeaway: OLS residuals are perpendicular to the column space of the design matrix.

### Basic 7 — Recompute the lesson training score

**Goal.** Average the three verified losses from the lesson text, because ERM is built from per-example losses. We build it in 2 steps.

In [ ]:
losses_b7 = np.array([0.235, 0.083, 0.454]) # Store the verified per-example losses.
total_b7 = float(np.sum(losses_b7)) # Sum losses before averaging.
print("loss total:", round(total_b7, 3)) # Inspect 0.772.
assert round(total_b7, 3) == 0.772 # Verify lesson total.

▶ What you'll see: three small losses combine to total 0.772.

In [ ]:
R_S_b7 = float(np.mean(losses_b7)) # Compute empirical average.
print("R_S:", round(R_S_b7, 3)) # Inspect training score.
assert round(R_S_b7, 3) == 0.257 # Verify lesson average.
plt.figure(figsize=(4, 3)) # Create loss breakdown.
plt.bar(["loss 1", "loss 2", "loss 3"], losses_b7, color="teal") # Show per-example losses.
plt.axhline(R_S_b7, color="red", linestyle="--", label="average") # Show ERM average.
plt.title("Basic 7: empirical risk average") # Title plot.
plt.legend() # Show average label.
plt.show() # Display.

▶ What you'll see: the average is pulled upward by the largest individual loss.

👀 Takeaway: the training number is an average over examples, not a single representative case.

### Basic 8 — Add the method cost

**Goal.** Add a 0.100 cost to the raw average, because the lesson's selection score is not just training fit. We build it in 2 steps.

In [ ]:
R_S_b8 = 0.257 # Use the rounded empirical risk from the lesson.
cost_b8 = 0.100 # Define the method cost or complexity penalty.
score_b8 = R_S_b8 + cost_b8 # Combine raw fit and cost.
print("score:", round(score_b8, 3)) # Inspect full decision score.
assert round(score_b8, 3) == 0.357 # Verify lesson score.

▶ What you'll see: the score increases after accounting for cost.

In [ ]:
plt.figure(figsize=(4, 3)) # Create comparison plot.
plt.bar(["raw R_S", "cost", "full score"], [R_S_b8, cost_b8, score_b8], color=["gray", "orange", "teal"]) # Show parts and total.
plt.title("Basic 8: fit plus cost") # Title plot.
plt.ylabel("score units") # Label score.
plt.show() # Display bars.

▶ What you'll see: the full score includes both the attractive raw fit and the guardrail cost.

👀 Takeaway: model selection must use the full score defined by the method.

### Basic 9 — Compare an alternative by gap

**Goal.** Compute absolute and relative score gaps, because small differences can be unstable under resampling noise. We build it in 2 steps.

In [ ]:
baseline_b9 = 0.357 # Full score for the baseline model.
flexible_b9 = 0.393 # Full score for a more flexible alternative.
gap_b9 = flexible_b9 - baseline_b9 # Compute absolute gap.
relative_b9 = gap_b9 / flexible_b9 # Compute relative gap on the alternative's scale.
print("gap:", round(gap_b9, 3), "relative:", round(relative_b9, 3)) # Inspect both comparisons.
assert round(gap_b9, 3) == 0.036 and round(relative_b9, 3) == 0.092 # Verify lesson values.

▶ What you'll see: the baseline wins, but the margin is only about 9.2% of the alternative score.

In [ ]:
plt.figure(figsize=(4, 3)) # Create score comparison.
plt.bar(["baseline", "flexible"], [baseline_b9, flexible_b9], color=["teal", "orange"]) # Plot comparable scores.
plt.title("Basic 9: comparable decision scores") # Title plot.
plt.ylabel("lower is better") # Label y.
plt.show() # Display comparison.

▶ What you'll see: the flexible alternative is higher, so it must overcome a real score gap to be preferred.

👀 Takeaway: score gaps are evidence; tiny gaps should be treated cautiously.

### Basic 10 — Apply the stabilization knob

**Goal.** Reduce the decision score by 20%, because the lesson uses stabilization to illustrate the flexibility-versus-durability trade. We build it in 2 steps.

In [ ]:
score_b10 = 0.357 # Baseline full score.
stable_b10 = 0.80 * score_b10 # Apply the 20% reduction.
print("stable score:", round(stable_b10, 3)) # Inspect stabilized score.
assert round(stable_b10, 3) == 0.286 # Verify lesson value.

▶ What you'll see: the stabilized score is lower than the baseline score.

In [ ]:
scores_b10 = np.array([score_b10, 0.393, stable_b10]) # Store baseline, flexible, stabilized.
labels_b10 = np.array(["baseline", "flexible", "stabilized"]) # Name each candidate.
best_b10 = labels_b10[int(np.argmin(scores_b10))] # Choose the lowest score.
print("best:", best_b10) # Inspect final decision.
assert best_b10 == "stabilized" # Verify lesson decision.
plt.figure(figsize=(4, 3)) # Create final comparison plot.
plt.bar(labels_b10, scores_b10, color=["teal", "orange", "seagreen"]) # Visualize all candidates.
plt.title("Basic 10: final minimum score") # Title plot.
plt.ylabel("decision score") # Label y.
plt.show() # Display.

▶ What you'll see: stabilized is the lowest bar, so it is the candidate to carry forward in this toy case.

👀 Takeaway: the final OLS-adjacent decision compares complete scores on one scale.

## 🟡 Easy

### Easy 1 — Fit OLS with `np.linalg.lstsq`

**Goal.** Use NumPy's least-squares solver, because it computes the same OLS solution without manually forming the normal equation. We build it in 3 steps.

In [ ]:
x_e1 = np.array([0., 1., 2., 3., 4.]) # Define feature values.
y_e1 = np.array([1.1, 2.0, 2.9, 4.1, 5.0]) # Define targets.
X_e1 = np.column_stack([np.ones_like(x_e1), x_e1]) # Build intercept-plus-slope design.
print("X shape:", X_e1.shape) # Inspect dimensions.

▶ What you'll see: five examples provide evidence for two coefficients.

In [ ]:
beta_e1, residuals_e1, rank_e1, singular_e1 = np.linalg.lstsq(X_e1, y_e1, rcond=None) # Fit OLS with a stable solver.
yhat_e1 = X_e1 @ beta_e1 # Compute fitted values.
print("beta:", np.round(beta_e1, 3), "rank:", rank_e1) # Inspect coefficients and rank.
assert np.allclose(np.round(beta_e1, 3), [1.040, 0.990]) # Verify coefficients.

▶ What you'll see: `lstsq` returns the same intercept and slope as the normal equation.

In [ ]:
plt.figure(figsize=(4, 3)) # Create fit plot.
plt.scatter(x_e1, y_e1, color="black", label="data") # Plot observed data.
plt.plot(x_e1, yhat_e1, color="teal", label="lstsq fit") # Plot fitted line.
plt.title("Easy 1: OLS with lstsq") # Title plot.
plt.xlabel("x") # Label x.
plt.ylabel("y") # Label y.
plt.legend() # Show legend.
plt.show() # Display.

▶ What you'll see: the least-squares line balances the points exactly like the manual solve.

👀 Takeaway: production code usually prefers a stable solver, but the math target is still least squares.

### Easy 2 — Fit multiple features

**Goal.** Extend OLS from one feature to two, because the design-matrix view scales by adding columns. We build it in 3 steps.

In [ ]:
size_e2 = np.array([1., 2., 3., 4., 5.]) # First feature.
age_e2 = np.array([5., 4., 3., 2., 1.]) # Second feature.
y_e2 = 2.0 + 1.5 * size_e2 - 0.5 * age_e2 # Construct a target from known coefficients.
X_e2 = np.column_stack([np.ones_like(size_e2), size_e2, age_e2]) # Build multi-feature design matrix.
print("X_e2 shape:", X_e2.shape) # Inspect rows and columns.

▶ What you'll see: OLS now estimates three coefficients: intercept, size effect, and age effect.

In [ ]:
beta_e2 = np.linalg.lstsq(X_e2, y_e2, rcond=None)[0] # Fit least squares.
print("beta:", np.round(beta_e2, 3)) # Inspect fitted coefficients.
yhat_e2 = X_e2 @ beta_e2 # Compute fitted values.
rmse_e2 = float(np.sqrt(np.mean((y_e2 - yhat_e2) ** 2))) # Compute training RMSE.
print("RMSE:", round(rmse_e2, 10)) # Inspect near-zero error.
assert rmse_e2 < 1e-10 # Verify exact fit for this constructed linear target.

▶ What you'll see: the fit predicts the constructed target essentially perfectly.

In [ ]:
plt.figure(figsize=(4, 3)) # Create coefficient plot.
plt.bar(["intercept", "size", "age"], beta_e2, color=["gray", "teal", "orange"]) # Show learned effects.
plt.title("Easy 2: learned coefficients") # Title plot.
plt.ylabel("coefficient") # Label coefficients.
plt.show() # Display.

▶ What you'll see: the learned coefficients are a valid least-squares solution, but correlated columns can make individual values non-unique.

👀 Takeaway: OLS handles many features by adding design columns, while rank determines whether coefficients are unique.

### Easy 3 — Compare train and validation error

**Goal.** Compute errors on held-out examples, because training loss alone does not prove reusable structure. We build it in 4 steps.

In [ ]:
x_train_e3 = np.array([0., 1., 2., 3., 4.]) # Training feature values.
y_train_e3 = np.array([1.1, 2.0, 2.9, 4.1, 5.0]) # Training targets.
X_train_e3 = np.column_stack([np.ones_like(x_train_e3), x_train_e3]) # Training design.
beta_e3 = np.linalg.lstsq(X_train_e3, y_train_e3, rcond=None)[0] # Fit OLS on training data.
print("beta:", np.round(beta_e3, 3)) # Inspect fitted rule.

▶ What you'll see: the model is fitted only from the training rows.

In [ ]:
train_pred_e3 = X_train_e3 @ beta_e3 # Predict training rows.
train_mse_e3 = float(np.mean((y_train_e3 - train_pred_e3) ** 2)) # Training MSE.
print("train MSE:", round(train_mse_e3, 4)) # Inspect training fit.
assert round(train_mse_e3, 4) == 0.0054 # Verify train error.

▶ What you'll see: the training error is very small.

In [ ]:
x_val_e3 = np.array([0.5, 1.5, 2.5, 3.5]) # Validation feature values.
y_val_e3 = np.array([1.55, 2.50, 3.45, 4.55]) # Validation targets.
X_val_e3 = np.column_stack([np.ones_like(x_val_e3), x_val_e3]) # Validation design.
val_pred_e3 = X_val_e3 @ beta_e3 # Predict validation rows.
val_mse_e3 = float(np.mean((y_val_e3 - val_pred_e3) ** 2)) # Validation MSE.
print("validation MSE:", round(val_mse_e3, 4)) # Inspect held-out score.
assert round(val_mse_e3, 4) == 0.0018 # Verify validation error.

▶ What you'll see: validation error is computed after fitting, not during fitting.

In [ ]:
plt.figure(figsize=(4, 3)) # Create error comparison.
plt.bar(["train", "validation"], [train_mse_e3, val_mse_e3], color=["teal", "orange"]) # Compare losses.
plt.title("Easy 3: train vs validation MSE") # Title plot.
plt.ylabel("MSE") # Label y.
plt.show() # Display.

▶ What you'll see: both scores are low here, but they are different measurements.

👀 Takeaway: validation checks whether the learned line behaves beyond the rows used to fit it.

### Easy 4 — Visualize residual diagnostics

**Goal.** Plot residuals against fitted values, because patterns in residuals reveal when a line is missing structure. We build it in 3 steps.

In [ ]:
x_e4 = np.array([-2., -1., 0., 1., 2.]) # Feature values.
y_e4 = 1.0 + 0.5 * x_e4 + 0.4 * x_e4 ** 2 # Curved target that a straight line cannot fully capture.
X_e4 = np.column_stack([np.ones_like(x_e4), x_e4]) # Fit only a straight-line design.
beta_e4 = np.linalg.lstsq(X_e4, y_e4, rcond=None)[0] # Fit linear OLS.
yhat_e4 = X_e4 @ beta_e4 # Compute fitted values.
resid_e4 = y_e4 - yhat_e4 # Compute residuals.
print("beta:", np.round(beta_e4, 3)) # Inspect best straight line.

▶ What you'll see: OLS finds a line even when the data were generated with curvature.

In [ ]:
print("residuals:", np.round(resid_e4, 3)) # Inspect the curved leftover pattern.
assert np.allclose(np.round(resid_e4, 3), [0.8, -0.4, -0.8, -0.4, 0.8]) # Verify residual pattern.

▶ What you'll see: residuals are positive at the ends and negative in the middle.

In [ ]:
plt.figure(figsize=(4, 3)) # Create residual diagnostic plot.
plt.scatter(yhat_e4, resid_e4, color="crimson") # Plot residuals against fitted values.
plt.axhline(0, color="black", linewidth=1) # Reference zero residual.
plt.title("Easy 4: residual pattern reveals curvature") # Title plot.
plt.xlabel("fitted value") # Label fitted axis.
plt.ylabel("residual") # Label residual axis.
plt.show() # Display.

▶ What you'll see: a U-shaped residual pattern, which says a straight line is systematically wrong.

👀 Takeaway: small code can expose when OLS assumptions are too simple for the data pattern.

### Easy 5 — Compare full decision scores

**Goal.** Recreate the lesson's full baseline, flexible, and stabilized comparison, because the final decision uses complete scores. We build it in 3 steps.

In [ ]:
losses_e5 = np.array([0.235, 0.083, 0.454]) # Verified losses.
raw_e5 = float(np.mean(losses_e5)) # Empirical risk.
cost_e5 = 0.100 # Method cost.
baseline_e5 = raw_e5 + cost_e5 # Full baseline score.
print("baseline:", round(baseline_e5, 3)) # Inspect baseline.
assert round(baseline_e5, 3) == 0.357 # Verify baseline.

▶ What you'll see: the baseline score includes both fit and cost.

In [ ]:
flexible_e5 = 0.393 # Alternative full score.
stabilized_e5 = 0.80 * baseline_e5 # Stabilized score.
scores_e5 = np.array([baseline_e5, flexible_e5, stabilized_e5]) # Candidate scores.
labels_e5 = np.array(["baseline", "flexible", "stabilized"]) # Candidate names.
print("scores:", np.round(scores_e5, 3)) # Inspect comparable scores.
assert np.allclose(np.round(scores_e5, 3), [0.357, 0.393, 0.286]) # Verify lesson scores.

▶ What you'll see: all candidates are now on the same score scale.

In [ ]:
best_e5 = labels_e5[int(np.argmin(scores_e5))] # Choose minimum score.
print("selected:", best_e5) # Inspect decision.
assert best_e5 == "stabilized" # Verify final selection.
plt.figure(figsize=(4, 3)) # Create final score plot.
plt.bar(labels_e5, scores_e5, color=["teal", "orange", "seagreen"]) # Visualize choices.
plt.title("Easy 5: final score comparison") # Title plot.
plt.ylabel("lower is better") # Label y.
plt.show() # Display.

▶ What you'll see: the stabilized score is the lowest of the three.

👀 Takeaway: a model-selection decision is only meaningful after raw fit, cost, and alternatives are compared consistently.

## 🔴 Advanced

### Advanced 1 — Handle rank deficiency with a pseudoinverse

**Goal.** Compare `solve` with a pseudoinverse when design columns are redundant, because $X^TX$ may not be invertible. We build it in 4 steps.

In [ ]:
x_a1 = np.array([0., 1., 2., 3.]) # Feature values.
X_a1 = np.column_stack([np.ones_like(x_a1), x_a1, 2 * x_a1]) # Third column duplicates the second column's information.
y_a1 = 1.0 + 3.0 * x_a1 # Linear target.
XtX_a1 = X_a1.T @ X_a1 # Normal-equation matrix.
print("rank of X:", np.linalg.matrix_rank(X_a1), "columns:", X_a1.shape[1]) # Inspect rank deficiency.

▶ What you'll see: the design has 3 columns but rank 2, so columns are redundant.

In [ ]:
cond_a1 = np.linalg.cond(XtX_a1) # Condition number of X^T X.
print("condition number:", cond_a1) # Inspect numerical singularity.
assert np.linalg.matrix_rank(X_a1) == 2 # Verify rank deficiency.

▶ What you'll see: the normal-equation matrix is too ill-conditioned for an ordinary inverse-based solve.

In [ ]:
beta_a1 = np.linalg.pinv(X_a1) @ y_a1 # Minimum-norm least-squares solution.
yhat_a1 = X_a1 @ beta_a1 # Predictions from pseudoinverse coefficients.
print("beta from pinv:", np.round(beta_a1, 3)) # Inspect one valid coefficient vector.
print("RMSE:", round(float(np.sqrt(np.mean((y_a1 - yhat_a1) ** 2))), 10)) # Inspect fit quality.
assert np.allclose(yhat_a1, y_a1) # Verify exact predictions despite non-unique coefficients.

▶ What you'll see: coefficients are not unique, but predictions can still be exactly right.

In [ ]:
plt.figure(figsize=(4, 3)) # Create coefficient plot.
plt.bar(["1", "x", "2x"], beta_a1, color=["gray", "teal", "orange"]) # Show split slope across redundant columns.
plt.title("Advanced 1: one minimum-norm solution") # Title plot.
plt.ylabel("coefficient") # Label coefficients.
plt.show() # Display.

▶ What you'll see: the slope is shared across the redundant x and 2x columns.

👀 Takeaway: the normal equation needs full rank for a unique coefficient vector; pseudoinverse keeps the least-squares prediction well-defined.

### Advanced 2 — Sweep ridge-style stabilization

**Goal.** Add $\lambda I$ before solving, because a stability penalty shrinks coefficients when features are noisy or nearly redundant. We build it in 4 steps.

In [ ]:
x_a2 = np.array([0., 1., 2., 3., 4., 5.]) # Feature values.
y_a2 = np.array([1.0, 2.2, 2.8, 4.4, 4.9, 6.2]) # Slightly noisy targets.
X_a2 = np.column_stack([np.ones_like(x_a2), x_a2]) # Design matrix.
lams_a2 = np.array([0.0, 0.1, 1.0, 10.0]) # Stabilization strengths.
print("lambda grid:", lams_a2) # Inspect sweep.

▶ What you'll see: the sweep ranges from ordinary OLS to strong shrinkage.

In [ ]:
betas_a2 = [] # Store coefficients.
train_mse_a2 = [] # Store training errors.
I_a2 = np.eye(X_a2.shape[1]) # Identity matrix for ridge-style penalty.
I_a2[0, 0] = 0.0 # Do not penalize the intercept in this demonstration.
for lam_a2 in lams_a2:
    beta_a2 = np.linalg.solve(X_a2.T @ X_a2 + lam_a2 * I_a2, X_a2.T @ y_a2) # Stabilized solve.
    pred_a2 = X_a2 @ beta_a2 # Predictions.
    betas_a2.append(beta_a2) # Store coefficients.
    train_mse_a2.append(float(np.mean((y_a2 - pred_a2) ** 2))) # Store MSE.
print("slopes:", np.round(np.array(betas_a2)[:, 1], 3)) # Inspect shrinkage.

▶ What you'll see: stronger λ shrinks the slope toward zero.

In [ ]:
print("train MSE:", np.round(train_mse_a2, 3)) # Inspect fit cost of shrinkage.
assert train_mse_a2[-1] > train_mse_a2[0] # Verify strong shrinkage raises training error here.

▶ What you'll see: stabilization can sacrifice raw training fit.

In [ ]:
plt.figure(figsize=(5, 3)) # Create sweep plot.
plt.plot(lams_a2, train_mse_a2, marker="o", color="purple") # Plot MSE vs lambda.
plt.xscale("symlog") # Show zero and larger lambda values compactly.
plt.title("Advanced 2: stabilization raises raw training loss") # Title plot.
plt.xlabel("λ") # Label lambda.
plt.ylabel("training MSE") # Label error.
plt.show() # Display.

▶ What you'll see: the raw training score gets worse as the penalty dominates, which is the cost of stability.

👀 Takeaway: regularization is a tradeoff: lower variance and smaller coefficients can come with higher training error.

### Advanced 3 — Bootstrap the validation gap

**Goal.** Resample validation errors to estimate uncertainty in a score gap, because a tiny gap may not be stable. We build it in 4 steps.

In [ ]:
errors_model_a3 = np.array([0.05, 0.08, 0.12, 0.04, 0.10, 0.07]) # Validation losses for one model.
errors_alt_a3 = np.array([0.03, 0.09, 0.10, 0.06, 0.17, 0.08]) # Validation losses for alternative.
gap_obs_a3 = float(np.mean(errors_alt_a3) - np.mean(errors_model_a3)) # Positive means model is lower loss.
print("observed gap:", round(gap_obs_a3, 3)) # Inspect average-loss gap.
assert round(gap_obs_a3, 3) == 0.012 # Verify tiny gap.

▶ What you'll see: the observed advantage is small.

In [ ]:
rng_a3 = np.random.default_rng(0) # Reproducible bootstrap generator.
boot_gaps_a3 = [] # Store bootstrap gaps.
for _ in range(1000):
    idx_a3 = rng_a3.integers(0, len(errors_model_a3), size=len(errors_model_a3)) # Resample validation rows.
    boot_gaps_a3.append(float(np.mean(errors_alt_a3[idx_a3]) - np.mean(errors_model_a3[idx_a3]))) # Store resampled gap.
boot_gaps_a3 = np.array(boot_gaps_a3) # Convert to array.
print("bootstrap mean gap:", round(float(np.mean(boot_gaps_a3)), 3)) # Inspect resampled average.

▶ What you'll see: resampling produces many plausible gaps near the observed one.

In [ ]:
lo_a3, hi_a3 = np.percentile(boot_gaps_a3, [5, 95]) # Compute a simple 90% interval.
print("90% interval:", round(float(lo_a3), 3), "to", round(float(hi_a3), 3)) # Inspect uncertainty.
assert lo_a3 < 0 < hi_a3 # Verify the interval crosses zero.

▶ What you'll see: zero is plausible, so the apparent winner is not decisive.

In [ ]:
plt.figure(figsize=(5, 3)) # Create bootstrap histogram.
plt.hist(boot_gaps_a3, bins=24, color="teal", edgecolor="white") # Plot gap distribution.
plt.axvline(0, color="black", linestyle="--", label="no gap") # Mark no-difference line.
plt.axvline(gap_obs_a3, color="red", label="observed") # Mark observed gap.
plt.title("Advanced 3: validation gap uncertainty") # Title plot.
plt.xlabel("alternative loss - model loss") # Label gap.
plt.legend() # Show reference lines.
plt.show() # Display.

▶ What you'll see: many resampled gaps sit close to zero, which warns against over-reading a tiny win.

👀 Takeaway: validation gaps are evidence with uncertainty, not decorations after training.

### Advanced 4 — Weighted least squares for unequal reliability

**Goal.** Fit OLS with per-example weights, because some observations may be more reliable than others. We build it in 4 steps.

In [ ]:
x_a4 = np.array([0., 1., 2., 3., 4.]) # Feature values.
y_a4 = np.array([1.0, 2.0, 3.1, 4.0, 8.0]) # Last point is noisy or low-reliability.
w_a4 = np.array([1.0, 1.0, 1.0, 1.0, 0.1]) # Down-weight the unreliable last point.
X_a4 = np.column_stack([np.ones_like(x_a4), x_a4]) # Design matrix.
print("weights:", w_a4) # Inspect reliability weights.

▶ What you'll see: the final example still counts, but much less than the others.

In [ ]:
beta_unweighted_a4 = np.linalg.lstsq(X_a4, y_a4, rcond=None)[0] # Ordinary OLS.
W_a4 = np.diag(w_a4) # Diagonal weight matrix.
beta_weighted_a4 = np.linalg.solve(X_a4.T @ W_a4 @ X_a4, X_a4.T @ W_a4 @ y_a4) # Weighted least squares.
print("unweighted beta:", np.round(beta_unweighted_a4, 3)) # Inspect ordinary fit.
print("weighted beta:", np.round(beta_weighted_a4, 3)) # Inspect down-weighted fit.

▶ What you'll see: the unweighted line bends more toward the unreliable high point.

In [ ]:
pred_unweighted_a4 = X_a4 @ beta_unweighted_a4 # Ordinary predictions.
pred_weighted_a4 = X_a4 @ beta_weighted_a4 # Weighted predictions.
weighted_loss_a4 = float(np.sum(w_a4 * (y_a4 - pred_weighted_a4) ** 2)) # Weighted objective value.
print("weighted objective:", round(weighted_loss_a4, 3)) # Inspect weighted score.
assert round(float(beta_weighted_a4[1]), 3) < round(float(beta_unweighted_a4[1]), 3) # Verify lower weighted slope.

▶ What you'll see: the weighted fit trusts the first four points more than the noisy last point.

In [ ]:
plt.figure(figsize=(4.8, 3.2)) # Create fit comparison plot.
plt.scatter(x_a4, y_a4, s=80 * w_a4 + 20, color="black", label="data (size=weight)") # Plot data with weight-sized markers.
plt.plot(x_a4, pred_unweighted_a4, color="orange", label="unweighted") # Plot ordinary fit.
plt.plot(x_a4, pred_weighted_a4, color="teal", label="weighted") # Plot weighted fit.
plt.title("Advanced 4: reliability weights change OLS") # Title plot.
plt.xlabel("x") # Label x.
plt.ylabel("y") # Label y.
plt.legend() # Show fit labels.
plt.show() # Display.

▶ What you'll see: the weighted line is less pulled upward by the small-weight outlier.

👀 Takeaway: changing the loss weights changes what “best fit” means, even with the same linear model family.

### Advanced 5 — Polynomial features and overfitting

**Goal.** Compare linear and high-degree polynomial designs, because adding columns can improve training fit while hurting future behavior. We build it in 5 steps.

In [ ]:
x_a5 = np.linspace(-1, 1, 9) # Training x-values.
y_a5 = 1.0 + 0.8 * x_a5 + 0.15 * np.sin(8 * x_a5) # Slightly wiggly targets.
X_lin_a5 = np.column_stack([np.ones_like(x_a5), x_a5]) # Linear design.
X_poly_a5 = np.column_stack([x_a5 ** k for k in range(7)]) # Degree-6 polynomial design.
print("linear columns:", X_lin_a5.shape[1], "poly columns:", X_poly_a5.shape[1]) # Inspect flexibility.

▶ What you'll see: the polynomial has many more columns and therefore much more flexibility.

In [ ]:
beta_lin_a5 = np.linalg.lstsq(X_lin_a5, y_a5, rcond=None)[0] # Fit linear model.
beta_poly_a5 = np.linalg.lstsq(X_poly_a5, y_a5, rcond=None)[0] # Fit flexible polynomial.
train_lin_a5 = float(np.mean((y_a5 - X_lin_a5 @ beta_lin_a5) ** 2)) # Linear training MSE.
train_poly_a5 = float(np.mean((y_a5 - X_poly_a5 @ beta_poly_a5) ** 2)) # Polynomial training MSE.
print("train MSE linear/poly:", round(train_lin_a5, 4), round(train_poly_a5, 4)) # Inspect raw fit.
assert train_poly_a5 < train_lin_a5 # Verify flexible model fits training better.

▶ What you'll see: the polynomial wins on training loss.

In [ ]:
x_val_a5 = np.linspace(-0.9, 0.9, 8) # Validation x-values between training points.
y_val_a5 = 1.0 + 0.8 * x_val_a5 + 0.15 * np.sin(8 * x_val_a5) # Validation targets from same rule.
X_val_lin_a5 = np.column_stack([np.ones_like(x_val_a5), x_val_a5]) # Linear validation design.
X_val_poly_a5 = np.column_stack([x_val_a5 ** k for k in range(7)]) # Polynomial validation design.
val_lin_a5 = float(np.mean((y_val_a5 - X_val_lin_a5 @ beta_lin_a5) ** 2)) # Linear validation MSE.
val_poly_a5 = float(np.mean((y_val_a5 - X_val_poly_a5 @ beta_poly_a5) ** 2)) # Polynomial validation MSE.
print("validation MSE linear/poly:", round(val_lin_a5, 4), round(val_poly_a5, 4)) # Inspect future behavior.

▶ What you'll see: validation decides whether the training improvement is reusable.

In [ ]:
grid_a5 = np.linspace(-1.1, 1.1, 200) # Dense grid for visualization.
Xg_lin_a5 = np.column_stack([np.ones_like(grid_a5), grid_a5]) # Linear grid design.
Xg_poly_a5 = np.column_stack([grid_a5 ** k for k in range(7)]) # Polynomial grid design.
line_a5 = Xg_lin_a5 @ beta_lin_a5 # Linear curve.
poly_a5 = Xg_poly_a5 @ beta_poly_a5 # Polynomial curve.
print("poly coefficient norm:", round(float(np.linalg.norm(beta_poly_a5)), 3)) # Inspect flexibility magnitude.

▶ What you'll see: the polynomial's coefficients can become large to chase wiggles.

In [ ]:
plt.figure(figsize=(5, 3.2)) # Create model comparison plot.
plt.scatter(x_a5, y_a5, color="black", label="train") # Plot training data.
plt.plot(grid_a5, line_a5, color="teal", label="linear") # Plot linear model.
plt.plot(grid_a5, poly_a5, color="crimson", label="degree 6") # Plot flexible model.
plt.title("Advanced 5: flexibility versus stability") # Title plot.
plt.xlabel("x") # Label x.
plt.ylabel("prediction") # Label y.
plt.legend() # Show curves.
plt.show() # Display.

▶ What you'll see: the flexible curve can chase training wiggles more aggressively than the straight line.

👀 Takeaway: extra columns reduce training loss, but validation and stabilization decide whether the flexibility is worth carrying forward.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

OLS chooses coefficients whose fitted line makes squared residuals as small as possible.

The notebook fits the same regression method from a hand line to a high-dimensional noisy D5. Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes, make_regression
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

np.random.seed(7)

def reg_ladder():
    """D1..D5 regression ladder of rising complexity. Returns [(name, X, y), ...]."""
    rungs = []
    x1 = np.array([[0.0], [1.0], [2.0], [3.0]])
    y1 = np.array([1.0, 3.0, 5.0, 7.0])
    rungs.append(("D1 hand line y=2x+1", x1, y1))
    rng = np.random.default_rng(1)
    x2 = np.linspace(-3, 3, 120).reshape(-1, 1)
    y2 = (2.0 * x2[:, 0] + 1.0) + rng.normal(0, 0.5, size=120)
    rungs.append(("D2 linear + noise", x2, y2))
    x3 = np.linspace(-3, 3, 160).reshape(-1, 1)
    y3 = np.sin(1.5 * x3[:, 0]) + rng.normal(0, 0.2, size=160)
    rungs.append(("D3 sine (non-linear)", x3, y3))
    dia = load_diabetes()
    rungs.append(("D4 Diabetes (real, 10-D)", dia.data, dia.target))
    x5, y5 = make_regression(n_samples=300, n_features=20, n_informative=8, noise=25.0, random_state=5)
    rungs.append(("D5 high-dim + noise (20-D)", x5, y5))
    return rungs

def reg_rmse(build_and_predict, X, y):
    """Split, call build_and_predict(x_tr, y_tr, x_te) -> preds, return held-out RMSE."""
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return float(np.sqrt(mean_squared_error(y_te, preds)))

def linear_baseline(x_tr, y_tr, x_te):
    clf = LinearRegression()
    clf.fit(x_tr, y_tr)
    return clf.predict(x_te)


## The concept, built once on D1

The lesson formula is $$ \hat\beta=(X^\top X)^{-1}X^\top y $$. The next cell recomputes the exact plan arithmetic before model fitting.

In [ ]:

def linear_regression_ols_normal_equation_method():
    losses = np.array([0.235, 0.083, 0.454], dtype=float)
    raw_sum = float(losses.sum())
    empirical_risk = round(float(raw_sum / len(losses)), 3)
    cost = 0.100
    score = round(empirical_risk + cost, 3)
    alternative = 0.393
    gap = round(alternative - score, 3)
    relative_gap = round(gap / alternative, 3)
    stable_score = round(0.80 * score, 3)
    final_score = min(score, alternative, stable_score)
    return {
        "losses": losses,
        "sum": raw_sum,
        "risk": empirical_risk,
        "cost": cost,
        "score": score,
        "alternative": alternative,
        "gap": gap,
        "relative_gap": relative_gap,
        "stable": stable_score,
        "final": final_score,
    }

lesson_check = linear_regression_ols_normal_equation_method()
print("losses:", lesson_check["losses"])
print("R_S =", round(lesson_check["sum"], 3), "/ 3 =", round(lesson_check["risk"], 3))
print("score =", round(lesson_check["score"], 3))
print("gap =", round(lesson_check["gap"], 3))
print("relative gap =", round(lesson_check["relative_gap"], 3))
print("stable score =", round(lesson_check["stable"], 3))
assert np.isclose(round(lesson_check["sum"], 3), 0.772)
assert np.isclose(round(lesson_check["risk"], 3), 0.257)
assert np.isclose(round(lesson_check["score"], 3), 0.357)
assert np.isclose(round(lesson_check["gap"], 3), 0.036)
assert np.isclose(round(lesson_check["relative_gap"], 3), 0.092)
assert np.isclose(round(lesson_check["stable"], 3), 0.286)


The exact assertions make the score, gap, and stabilized decision match the lesson.

In [ ]:

def design_with_intercept(X):
    return np.hstack([np.ones((X.shape[0], 1)), X])

def normal_equation_fit(X, y, ridge=0.0):
    X_design = design_with_intercept(X)
    gram = X_design.T.dot(X_design)
    penalty = ridge * np.eye(gram.shape[0])
    penalty[0, 0] = 0.0
    beta = np.linalg.pinv(gram + penalty).dot(X_design.T).dot(y)
    return beta

def normal_equation_predict(beta, X):
    return design_with_intercept(X).dot(beta)

def ols_predictor(x_tr, y_tr, x_te):
    beta = normal_equation_fit(x_tr, y_tr)
    return normal_equation_predict(beta, x_te)

def gradient_descent_fit(X, y, lr=0.03, steps=2500):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X_design = design_with_intercept(X_scaled)
    beta = np.zeros(X_design.shape[1])
    losses = []
    for step in range(steps):
        residual = X_design.dot(beta) - y
        grad = (2.0 / len(y)) * X_design.T.dot(residual)
        beta = beta - lr * grad
        if step % 50 == 0 or step == steps - 1:
            losses.append(float(np.mean(residual ** 2)))
    return beta, scaler, losses

def gradient_descent_predict(beta, scaler, X):
    X_scaled = scaler.transform(X)
    return design_with_intercept(X_scaled).dot(beta)

def gd_predictor(x_tr, y_tr, x_te):
    beta, scaler, losses = gradient_descent_fit(x_tr, y_tr, lr=0.03, steps=2500)
    return gradient_descent_predict(beta, scaler, x_te)

def run_regression_ladder(kind):
    rows = []
    for rung, (name, X, y) in enumerate(reg_ladder(), start=1):
        x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0)
        if kind == "ols":
            beta = normal_equation_fit(x_tr, y_tr)
            pred = normal_equation_predict(beta, x_te)
            history = []
        else:
            beta, scaler, history = gradient_descent_fit(x_tr, y_tr)
            pred = gradient_descent_predict(beta, scaler, x_te)
        mse = float(mean_squared_error(y_te, pred))
        rows.append({
            "rung": rung,
            "name": name,
            "n": X.shape[0],
            "d": X.shape[1],
            "metric": mse,
            "rmse": float(np.sqrt(mse)),
            "x_te": x_te,
            "y_te": y_te,
            "pred": pred,
            "history": history,
        })
    return rows

def plot_regression_panel(ax, X, y, pred, title):
    if X.shape[1] == 1:
        ax.scatter(X[:, 0], y, s=14, alpha=0.6)
        order = np.argsort(X[:, 0])
        ax.plot(X[order, 0], pred[order], color="crimson", linewidth=2)
    else:
        ax.scatter(y, pred, s=14, alpha=0.6)
        low = min(float(np.min(y)), float(np.min(pred)))
        high = max(float(np.max(y)), float(np.max(pred)))
        ax.plot([low, high], [low, high], color="crimson", linewidth=1)
    ax.set_title(title, fontsize=8)
    ax.tick_params(labelsize=7)


## The dataset ladder

The shared `reg_ladder` supplies D1–D5, ending with a real high-dimensional noisy regression stress test.

In [ ]:

rungs = reg_ladder()
for name, X, y in rungs:
    preview = np.round(X[:3, :min(4, X.shape[1])], 3)
    print(name)
    print("  shape:", X.shape)
    print("  target range:", (round(float(np.min(y)), 3), round(float(np.max(y)), 3)))
    print("  sample columns:")
    print(preview)


## Run the same regression method across D1–D5

The plan metric is MSE; RMSE is printed only as a readable secondary annotation.

In [ ]:

results = run_regression_ladder("ols")
print("rung | MSE | RMSE | dimensions")
for row in results:
    print(f"D{row['rung']} | {row['metric']:.3f} | {row['rmse']:.3f} | {row['d']}")
helper_rmse = reg_rmse(ols_predictor, rungs[-1][1], rungs[-1][2])
print("D5 helper reg_rmse:", round(helper_rmse, 3))


## Results visualization

The closing figure shows fitted values or residual structure per rung, plus MSE over complexity.

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
flat_axes = axes.ravel()
for ax, row in zip(flat_axes[:5], results):
    plot_regression_panel(ax, row["x_te"], row["y_te"], row["pred"], f"D{row['rung']} MSE={row['metric']:.1f}")
flat_axes[5].plot([row["rung"] for row in results], [row["metric"] for row in results], marker="o")
flat_axes[5].set_xlabel("rung")
flat_axes[5].set_ylabel("MSE")
fig.tight_layout()
plt.show()


## Pitfall on D5: optimizing the raw term and forgetting the cost

For regression the raw MSE can favor a brittle high-flexibility fit; the fix adds the lesson cost and compares on one scale.

In [ ]:

d5 = results[-1]
raw_mse = d5["metric"]
cost = lesson_check["cost"]
fixed_score = raw_mse + cost * raw_mse
stabilized_score = 0.80 * fixed_score
print("D5:", d5["name"])
print("wrong raw MSE:", round(raw_mse, 3))
print("fixed MSE plus scaled cost:", round(fixed_score, 3))
print("stabilized score:", round(stabilized_score, 3))
assert fixed_score > raw_mse
assert stabilized_score < fixed_score


## Evaluate it + Practice

- Metric: MSE; compare against the mean-target baseline.
- Sanity check: residuals should not show a simple missed line on D1/D2.
- Ablation: remove scaling before gradient descent or remove the cost term before selection.
- Failure signals: exploding loss, residual trend, or D5 improvement only on the raw score.

Practice 1: Add a ridge value to the normal equation and rerun D5.

Practice 2: Change the gradient-descent learning rate and plot the loss curve.

Practice 3: Compute the mean-target baseline MSE for every rung.